In [ ]:

import xml.etree.ElementTree as ET
import pandas as pd

XML_PATH = "/content/drive/MyDrive/Datasets/full_database.xml"
NS       = {"db": "http://www.drugbank.ca"}

all_diabetic_wound_healing_drugs = [
    "Metformin",
    "Glibenclamide", "Glimepiride", "Gliclazide", "Glipizide",
    "Tolbutamide", "Chlorpropamide",
    "Repaglinide", "Nateglinide",
    "Acarbose", "Voglibose", "Miglitol",
    "Pioglitazone", "Rosiglitazone",
    "Sitagliptin", "Vildagliptin", "Saxagliptin", "Alogliptin", "Linagliptin",
    "Semaglutide", "Liraglutide", "Dulaglutide", "Exenatide", "Tirzepatide", "Albiglutide",
    "Empagliflozin", "Dapagliflozin", "Canagliflozin", "Ertugliflozin",
    "Insulin Glargine", "Insulin Detemir", "Insulin Aspart",
    "Insulin Lispro", "Insulin NPH", "Insulin",
    "Silver Sulfadiazine", "Mupirocin", "Fusidic Acid", "Bacitracin",
    "Neomycin", "Gentamicin", "Chlorhexidine", "Povidone-Iodine",
    "Amoxicillin-Clavulanate", "Ciprofloxacin", "Levofloxacin",
    "Moxifloxacin", "Metronidazole", "Clindamycin", "Doxycycline",
    "Vancomycin", "Linezolid", "Daptomycin", "Tedizolid",
    "Trimethoprim-Sulfamethoxazole", "Ceftriaxone", "Cephalexin",
    "Piperacillin-Tazobactam", "Meropenem", "Imipenem", "Ertapenem",
    "Nitrofurantoin", "Azithromycin", "Erythromycin", "Amoxicillin",
    "Fluconazole", "Itraconazole", "Voriconazole", "Amphotericin B",
    "Terbinafine", "Clotrimazole", "Miconazole",
    "Aspirin", "Clopidogrel", "Ticagrelor", "Prasugrel", "Dipyridamole", "Cilostazol",
    "Atorvastatin", "Rosuvastatin", "Simvastatin", "Pravastatin", "Fluvastatin", "Pitavastatin",
    "Fenofibrate", "Gemfibrozil", "Bezafibrate",
    "Ezetimibe", "Niacin",
    "Enalapril", "Lisinopril", "Ramipril", "Perindopril", "Captopril",
    "Losartan", "Telmisartan", "Valsartan", "Irbesartan", "Candesartan", "Olmesartan",
    "Metoprolol", "Carvedilol", "Bisoprolol", "Atenolol", "Nebivolol",
    "Amlodipine", "Nifedipine", "Diltiazem", "Verapamil",
    "Heparin", "Warfarin", "Apixaban", "Rivaroxaban", "Dabigatran", "Enoxaparin",
    "Digoxin", "Ivabradine", "Nitroglycerin", "Isosorbide Mononitrate",
    "Spironolactone", "Furosemide", "Torsemide", "Hydralazine",
    "Pregabalin", "Gabapentin", "Duloxetine", "Amitriptyline",
    "Nortriptyline", "Venlafaxine", "Carbamazepine", "Oxcarbazepine",
    "Tapentadol", "Tramadol", "Alpha-lipoic acid", "Benfotiamine",
    "Sevelamer", "Epoetin Alfa", "Darbepoetin", "Finerenone",
    "Ranibizumab", "Bevacizumab", "Aflibercept", "Brolucizumab",
    "Faricimab", "Timolol", "Latanoprost", "Brimonidine", "Acetazolamide",
    "Becaplermin", "Collagenase", "Pentoxifylline",
    "Vitamin C", "Vitamin E", "Vitamin D", "Zinc Sulfate", "Calcium", "Magnesium",
    "Metoclopramide", "Domperidone", "Midodrine", "Fludrocortisone",
    "Levothyroxine", "Methimazole", "Carbimazole",
    "Alendronate", "Risedronate", "Zoledronic acid", "Denosumab", "Teriparatide",
    "Sertraline", "Fluoxetine", "Escitalopram", "Paroxetine",
    "Mirtazapine", "Bupropion", "Memantine", "Donepezil", "Rivastigmine",
    "Modafinil", "Melatonin", "Zolpidem",
    "Clomiphene",
    "Paracetamol", "Ibuprofen", "Diclofenac", "Naproxen",
    "Celecoxib", "Prednisolone", "Dexamethasone", "Methylprednisolone", "Hydrocortisone",
    "Isoniazid", "Rifampicin", "Ethambutol", "Pyrazinamide",
    "Sildenafil", "Tadalafil",
]

drug_set = {d.lower() for d in all_diabetic_wound_healing_drugs}


def map_severity(description):
    if not description:
        return "unknown"
    desc = description.lower()
    if any(w in desc for w in ["serious", "contraindicated", "avoid",
                                "life-threatening", "major", "severe",
                                "fatal", "toxicity"]):
        return "strong"
    elif any(w in desc for w in ["monitor", "caution", "moderate",
                                  "adjust dose", "reduce dose", "increase"]):
        return "moderate"
    else:
        return "weak"


def get_smiles(drug):
    calc_props = drug.find("db:calculated-properties", NS)
    if calc_props:
        for prop in calc_props.findall("db:property", NS):
            if prop.findtext("db:kind", namespaces=NS) == "SMILES":
                return prop.findtext("db:value", namespaces=NS)
    return None


print("Parsing XML...")
tree = ET.parse(XML_PATH)
root = tree.getroot()

smiles_lookup = {}

for drug in root.findall("db:drug", NS):
    name = drug.findtext("db:name", namespaces=NS)
    if not name or name.lower() not in drug_set:
        continue
    smiles_lookup[name.lower()] = {
        "name"   : name,
        "smiles" : get_smiles(drug)
    }

print(f"Drugs found     : {len(smiles_lookup)}")
print(f"Drugs with SMILES: {sum(1 for v in smiles_lookup.values() if v['smiles'])}")

rows = []

for drug in root.findall("db:drug", NS):
    name1 = drug.findtext("db:name", namespaces=NS)
    if not name1 or name1.lower() not in smiles_lookup:
        continue

    ddi_block = drug.find("db:drug-interactions", NS)
    if ddi_block is None:
        continue

    for interaction in ddi_block.findall("db:drug-interaction", NS):
        name2 = interaction.findtext("db:name", namespaces=NS)
        if not name2 or name2.lower() not in smiles_lookup:
            continue

        description = interaction.findtext("db:description", namespaces=NS)
        severity    = map_severity(description)

        rows.append({
            "drug1_name"             : name1,
            "drug1_smiles"           : smiles_lookup[name1.lower()]["smiles"],
            "drug2_name"             : name2,
            "drug2_smiles"           : smiles_lookup[name2.lower()]["smiles"],
            "interaction_description": description,
            "severity"               : severity,
        })

df = pd.DataFrame(rows)

print(f"Total pairs      : {len(df)}")
print(f"Unique drug1     : {df['drug1_name'].nunique()}")
print(f"Unique drug2     : {df['drug2_name'].nunique()}")
print(f"\nSeverity distribution:")
print(df["severity"].value_counts())

df.to_csv("/content/drive/MyDrive/Datasets/ddi_diabetic_wound_healing_w_smiles.csv", index=False)
print("\nSaved!")


Parsing XML...


KeyboardInterrupt: 

In [ ]:
import pandas as pd

tt = pd.read_csv("/content/drive/MyDrive/Datasets/ddi_diabetic_wound_healing_w_smiles.csv")
ttt = tt.dropna()
ttt.to_csv("/content/drive/MyDrive/Datasets/ddi_diabetic_wound_healing_w_smiles_dropna().csv")

In [ ]:
!pip install rdkit
!pip install cirpy

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
fin = pd.read_csv("/content/drive/MyDrive/Datasets/ddi_diabetic_wound_healing_w_smiles_dropna().csv")
fin

,Unnamed: 0,drug1_name,drug1_smiles,drug2_name,drug2_smiles,interaction_description,severity
0,172,Daptomycin,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,Pravastatin,[H][C@]12[C@H](C[C@H](O)C=C1C=C[C@H](C)[C@@H]2...,The risk or severity of myopathy can be increa...,moderate
1,173,Daptomycin,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,Simvastatin,[H][C@]12[C@H](C[C@@H](C)C=C1C=C[C@H](C)[C@@H]...,The risk or severity of myopathy can be increa...,moderate
2,174,Daptomycin,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,Atorvastatin,CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CC[C@@H](O)...,The risk or severity of myopathy can be increa...,moderate
3,175,Daptomycin,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,Fluvastatin,CC(C)N1C(\C=C\C(O)CC(O)CC(O)=O)=C(C2=CC=CC=C12...,The risk or severity of myopathy can be increa...,moderate
4,176,Daptomycin,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,Rosuvastatin,CC(C)C1=NC(=NC(C2=CC=C(F)C=C2)=C1\C=C\[C@@H](O...,The risk or severity of myopathy can be increa...,moderate
...,...,...,...,...,...,...,...
12769,14669,Finerenone,CCOC1=NC=C(C)C2=C1[C@H](C1=CC=C(C=C1OC)C#N)C(C...,Prasugrel,CC(=O)OC1=CC2=C(CCN(C2)C(C(=O)C2CC2)C2=CC=CC=C...,Finerenone may increase the excretion rate of ...,moderate
12770,14670,Finerenone,CCOC1=NC=C(C)C2=C1[C@H](C1=CC=C(C=C1OC)C#N)C(C...,Rivaroxaban,ClC1=CC=C(S1)C(=O)NC[C@H]1CN(C(=O)O1)C1=CC=C(C...,Finerenone may increase the excretion rate of ...,moderate
12771,14671,Finerenone,CCOC1=NC=C(C)C2=C1[C@H](C1=CC=C(C=C1OC)C#N)C(C...,Saxagliptin,[H][C@@]12C[C@]1([H])N([C@@H](C2)C#N)C(=O)[C@@...,Finerenone may increase the excretion rate of ...,moderate
12772,14672,Finerenone,CCOC1=NC=C(C)C2=C1[C@H](C1=CC=C(C=C1OC)C#N)C(C...,Tapentadol,CC[C@H]([C@@H](C)CN(C)C)C1=CC(O)=CC=C1,The therapeutic efficacy of Finerenone can be ...,weak


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator


gen = GetMorganGenerator(radius = 1, fpSize = 1024)
def smile_finger_print(smile):
  mol = Chem.MolFromSmiles(smile)
  if mol:
    fp = gen.GetFingerprintAsNumPy(mol)
    return list(fp)
  else:
    print("error")

fin['smile_fp1'] = fin['drug1_smiles'].apply(smile_finger_print)
fin['smile_fp2'] = fin['drug2_smiles'].apply(smile_finger_print)



In [ ]:
fin.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.model_selection import train_test_split
le = LabelEncoder()


fp1 = np.array(fin['smile_fp1'].tolist())
fp2 = np.array(fin['smile_fp2'].tolist())

x = np.hstack([fp1, fp2])
y = le.fit_transform(fin['severity'])


xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [ ]:
#severity prediction using ANN
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

my_mod = Sequential(
    [
        Input(shape = (x.shape[1],)),
        Dense(16, activation = 'relu'),
        Dropout(0.5),

        Dense(32, activation = 'relu'),
        Dropout(0.5),

        Dense(64, activation = 'relu'),
        Dropout(0.5),

        Dense(3, activation = 'softmax')

    ]
)

my_mod.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

my_mod.fit(xtrain, ytrain, epochs = 100, batch_size = 32, validation_split = 0.2)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

loss, accuracy = my_mod.evaluate(xtest, ytest)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")


y_pred_probs = my_mod.predict(xtest)
y_pred = np.argmax(y_pred_probs, axis=1)

acc = accuracy_score(ytest, y_pred)
prec = precision_score(ytest, y_pred, average='weighted')
rec = recall_score(ytest, y_pred, average='weighted')

print(f"\nAccuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")


cm = confusion_matrix(ytest, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
my_mod.save("/content/drive/MyDrive/Datasets/ddi_model.keras")

In [ ]:
my_mod.save_weights('/content/drive/MyDrive/Datasets/ddi_model_weights.weights.h5')

In [ ]:
yp = my_mod.predict(xtest)
print(yp)
predicted_severity_labels = le.inverse_transform(np.argmax(yp, axis=1))

print("\nPredicted Severity Labels (first 10):\n", predicted_severity_labels[:10])


In [ ]:
#custom input
import cirpy
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import tensorflow as tf


da = input("Drug A: ")
db = input("Drug B: ")

smilea = cirpy.resolve(da, 'smiles')
smileb = cirpy.resolve(db, 'smiles')

print(f"SMILES for {da}: {smilea}")
print(f"SMILES for {db}: {smileb}")

# Check if SMILES strings were resolved
if smilea is None:
    print(f"Error: Could not resolve SMILES for {da}. Please check the drug name.")
elif smileb is None:
    print(f"Error: Could not resolve SMILES for {db}. Please check the drug name.")
else:
    #converting smile to morgan finger print (as a bit vector)

    gen = GetMorganGenerator(radius = 1, fpSize = 1024)
    def smile_finger_print(smile):
        mol = Chem.MolFromSmiles(smile)
        if mol:
            fp = gen.GetFingerprintAsNumPy(mol)
            return list(fp)
        else:
            # This case should ideally not be reached if smile is not None
            # but it's good for robustness.
            print(f"Error: Could not create Mol object from SMILES: {smile}")
            return None

    fpa = smile_finger_print(smilea)
    fpb = smile_finger_print(smileb)

    # Ensure fingerprints were successfully generated
    if fpa is None or fpb is None:
        print("Error: Could not generate fingerprints for one or both drugs.")
    else:
        fpa = np.array(fpa)
        fpb = np.array(fpb)

        train_df = pd.DataFrame()
        train_df["fp1"] = [fpa]
        train_df["fp2"] = [fpb]

        model = tf.keras.models.load_model("/content/drive/MyDrive/Datasets/ddi_model.keras")

        # Combine fingerprints for prediction
        combined_fp = np.hstack([fpa.reshape(1, -1), fpb.reshape(1, -1)])
        y_pred = model.predict(combined_fp)

        predicted_severity_label = le.inverse_transform(np.argmax(y_pred, axis=1))

        print("\nPredicted Severity Label:\n", predicted_severity_label)


In [ ]:
#drug effect
positive_keywords = [
    # Beneficial therapeutic effects
    "may increase the anticoagulant",
    "may increase the antihypertensive",
    "may increase the hypoglycemic",
    "may increase the antidiabetic",
    "may increase the diuretic",
    "may increase the antiplatelet",
    "may increase the hypolipidaemic",
    "may increase the vasodilatory",
    "may increase the hypotensive",
    "may decrease the nephrotoxic",
    "may increase the excretion",
    "may decrease the excretion",
    "therapeutic efficacy",
    "increase the therapeutic",
    "increase the efficacy",
    "beneficial",
    "improve",
    "enhance",
    "protective",
    "promote",
]

negative_keywords = [
    # Harmful / dangerous interactions
    "risk or severity",
    "adverse effects",
    "may increase the hepatotoxic",
    "may increase the neurotoxic",
    "may increase the nephrotoxic",
    "may increase the ototoxic",
    "may increase the myopathic",
    "may increase the arrhythmogenic",
    "may increase the bradycardic",
    "may increase the tachycardic",
    "may increase the hyperkalemic",
    "may increase the serotonergic",
    "may increase the neuroexcitatory",
    "may increase the qtc",
    "may increase the central",
    "may increase the orthostatic",
    "may increase the vasopressor",
    "rhabdomyolysis",
    "myoglobinuria",
    "myopathy",
    "hypoglycemia",
    "hyperkalemia",
    "hypotension",
    "bleeding",
    "toxicity",
    "toxic",
    "fatal",
    "contraindicated",
    "serious",
    "severe",
    "avoid",
    "danger",
    "failure",
    "renal failure",
]

neutral_keywords = [
    # Metabolism / pharmacokinetic changes with no clear harm or benefit
    "metabolism of",
    "serum concentration",
    "may decrease the antihypertensive",
    "may decrease the anticoagulant",
    "may decrease the hypoglycemic",
    "may decrease the excretion",
    "excretion rate",
    "serum level",
    "higher serum",
    "protein binding",
    "clearance",
    "absorption",
    "bioavailability",
    "half-life",
]

def drug_effect(interaction_description):
  for i in positive_keywords:
    if i in interaction_description:
      return 1
  for i in negative_keywords:
    if i in interaction_description:
      return -1
  for i in neutral_keywords:
    if i in interaction_description:
      return 0
  return 0

fin['drug_effect'] = fin['interaction_description'].apply(drug_effect)


In [ ]:
fin.to_csv("/content/drive/MyDrive/Datasets/with_interaction.csv")

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator


fin = pd.read_csv("/content/drive/MyDrive/Datasets/with_interaction.csv")

gen = GetMorganGenerator(radius = 1, fpSize = 1024)
def smile_finger_print(smile):
  mol = Chem.MolFromSmiles(smile)
  if mol:
    fp = gen.GetFingerprintAsNumPy(mol)
    return list(fp)
  else:
    print("error")

fin['smile_fp1'] = fin['drug1_smiles'].apply(smile_finger_print)
fin['smile_fp2'] = fin['drug2_smiles'].apply(smile_finger_print)

fin.to_csv("/content/drive/MyDrive/Datasets/with_interaction_and_fingerprint.csv")

In [ ]:
dp = pd.read_csv("/content/drive/MyDrive/Datasets/with_interaction_and_fingerprint.csv")

x

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
my_mod = Sequential(
    [
        Input(shape = (x.shape[1],)),
        Dense(16, activation = 'relu'),
        Dropout(0.5),

        Dense(32, activation = 'relu'),
        Dropout(0.5),

        Dense(64, activation = 'relu'),
        Dropout(0.5),

        Dense(3, activation = 'softmax')server

    ]
)

my_mod.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

my_mod.fit(xtrain, ytrain, epochs = 100, batch_size = 32, validation_split = 0.2)